# Pythonic recipe builder example

This notebook shows how to author a Woodpecker recipe in Python and serialize it to the same JSON/YAML recipe-document schema used by stores, catalogs, and the CLI.

The builder API is useful when JSON or YAML feels too verbose for normal authoring, but you still want the generated recipe document to be portable.

In [ ]:
import numpy as np

import woodpecker
from woodpecker.recipes import document, fix, recipe
from woodpecker.testing import make_cmip6

Build a recipe using normal Python calls. A `fix(...)` is a fix-function id plus optional options; `recipe(...)` collects ordered fixes and recipe metadata.

In [ ]:
cmip6_core = recipe(
    "cmip6.core_units",
    fix("woodpecker.normalize_tas_units_to_kelvin"),
    description="Normalize CMIP6 tas units.",
).match(
    dataset_id_patterns=["CMIP6.CMIP.*.Amon.tas.*"],
    attrs={"project_id": "CMIP6", "activity_id": "CMIP"},
)

cmip6_core.to_payload()

Serialize the Python-authored recipe to a recipe document. The same builder can write to a path when you pass one to `to_yaml(...)` or `to_json(...)`.

In [ ]:
print(cmip6_core.to_yaml())

Use `to_model()` when you want to run the recipe immediately through the normal Woodpecker recipe API.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)
original_values = dataset["tas"].values.copy()

plan_model = cmip6_core.to_model()
findings = woodpecker.recipe.check(dataset, plan_model)

findings.fix_ids

In [ ]:
result = woodpecker.recipe.fix(dataset, plan_model, dry_run=True)

result.stats, result.preview, dataset["tas"].attrs["units"], np.allclose(dataset["tas"].values, original_values)

In [ ]:
write = woodpecker.recipe.fix(dataset, plan_model, dry_run=False)

(
    write.stats,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values + 273.15),
)

A recipe document can contain multiple Python-authored recipes. This is the shape you would serialize for a package resource or a shared recipe file.

In [ ]:
plan_document = document(
    cmip6_core,
    recipe("atlas.basic", fix("atlas.encoding_cleanup")).match(path_patterns=["*atlas*.nc"]),
)

plan_document.to_payload()